In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

print("TensorFlow version:", tf.__version__)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(
    num_words=10000
)

def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

print("Training samples:", len(partial_x_train))
print("Validation samples:", len(x_val))
print("Test samples:", len(x_test))
print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)


The function below makes it easier to run the same experiment with different numbers of hidden layers, hidden units, activation functions, loss functions, dropout, and L2 regularization.

In [ ]:
def run_model(
    name,
    hidden_units=(16, 16),
    activation="relu",
    loss="binary_crossentropy",
    dropout_rate=None,
    l2_rate=None,
    epochs=10,
    verbose=1
):
    model = keras.Sequential()
    model.add(layers.Input(shape=(10000,)))

    for units in hidden_units:
        kwargs = {"activation": activation}
        if l2_rate is not None:
            kwargs["kernel_regularizer"] = regularizers.l2(l2_rate)
        model.add(layers.Dense(units, **kwargs))
        if dropout_rate is not None:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer="rmsprop", loss=loss, metrics=["accuracy"])

    history = model.fit(
        partial_x_train,
        partial_y_train,
        epochs=epochs,
        batch_size=512,
        validation_data=(x_val, y_val),
        verbose=verbose
    )

    best_epoch = int(np.argmax(history.history["val_accuracy"])) + 1
    best_val = float(max(history.history["val_accuracy"]))
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

    return {
        "Model": name,
        "Hidden Layers": len(hidden_units),
        "Hidden Units": ", ".join(map(str, hidden_units)),
        "Change": name,
        "Best Validation Accuracy": best_val,
        "Best Epoch": best_epoch,
        "Test Accuracy": float(test_accuracy),
        "History": history.history
    }



The baseline uses two hidden layers with 16 units in each layer. The assignment asks for comparisons using one and three hidden layers and different numbers of hidden units.

In [ ]:
results = []

experiments = [
    ("One Hidden Layer", (16,), "relu", "binary_crossentropy", None, None),
    ("Baseline - 2 Layers", (16, 16), "relu", "binary_crossentropy", None, None),
    ("Three Hidden Layers", (16, 16, 16), "relu", "binary_crossentropy", None, None),
    ("32 Units", (32, 32), "relu", "binary_crossentropy", None, None),
    ("64 Units", (64, 64), "relu", "binary_crossentropy", None, None),
]

for exp in experiments:
    result = run_model(*exp, verbose=1)
    results.append(result)
    print(f"{result['Model']}: validation={result['Best Validation Accuracy']:.4f}, test={result['Test Accuracy']:.4f}")


MSE loss, tanh activation, dropout, and  regularization.

In [ ]:
more_experiments = [
    ("MSE Loss", (16, 16), "relu", "mse", None, None),
    ("Tanh Activation", (16, 16), "tanh", "binary_crossentropy", None, None),
    ("Dropout 0.2", (16, 16), "relu", "binary_crossentropy", 0.2, None),
    ("Dropout 0.5", (16, 16), "relu", "binary_crossentropy", 0.5, None),
    ("L2 Regularization", (16, 16), "relu", "binary_crossentropy", None, 0.001),
]

for exp in more_experiments:
    result = run_model(*exp, verbose=1)
    results.append(result)
    print(f"{result['Model']}: validation={result['Best Validation Accuracy']:.4f}, test={result['Test Accuracy']:.4f}")


Validation accuracy is used to compare how the configurations performed during model development, while test accuracy provides the final performance on the held-out test set.

In [ ]:
results_df = pd.DataFrame(results)
display_cols = [
    "Model", "Hidden Layers", "Hidden Units", "Change",
    "Best Validation Accuracy", "Best Epoch", "Test Accuracy"
]

results_display = results_df[display_cols].copy()
results_display["Best Validation Accuracy"] *= 100
results_display["Test Accuracy"] *= 100

results_display.columns = [
    "Model", "Hidden Layers", "Hidden Units", "Change",
    "Best Validation Accuracy (%)", "Best Epoch", "Test Accuracy (%)"
]

display(results_display.round(2))
results_display.to_csv("IMDB_Neural_Network_Results.csv", index=False)

Validation accuracy comparison

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(results_display["Model"], results_display["Best Validation Accuracy (%)"])
plt.ylabel("Best Validation Accuracy (%)")
plt.xlabel("Model")
plt.title("IMDB Neural Network Validation Accuracy")
plt.xticks(rotation=45, ha="right")
plt.ylim(80, 92)
plt.tight_layout()
plt.show()

Test accuracy Comparison

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(results_display["Model"], results_display["Test Accuracy (%)"])
plt.ylabel("Test Accuracy (%)")
plt.xlabel("Model")
plt.title("IMDB Neural Network Test Accuracy")
plt.xticks(rotation=45, ha="right")
plt.ylim(80, 92)
plt.tight_layout()
plt.show()